In [ ]:
import os

import pandas as pd
import requests
from bs4 import BeautifulSoup

from open_parcel.scrapers.toronto_bylaw_parser import (
    extract_content_rows,
    extract_table_of_contents,
)

## downloading bylaw text


In [ ]:
base_url = "https://www.toronto.ca/zoning/bylaw_amendments/"
root_page = "ZBL_NewProvision_Chapter1.htm"
cache_path = "data/toronto/input/zoning_bylaw/ZBL_NewProvision_Chapter1.htm"

if os.path.exists(cache_path):
    print("file exists")
    with open(cache_path, encoding="utf-8") as file:
        html = file.read()
else:
    print("fetching file")
    response = requests.get(base_url + root_page)
    html = response.text
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as file:
        file.write(html)

soup = BeautifulSoup(html, "html.parser")

In [ ]:
tables = soup.find_all("table")
toc_index = next((i for i, x in enumerate(tables) if "Table of Contents" in x.text), None)

if toc_index is not None:
    toc = tables[toc_index]
    # left column - table of contents
    toc_data = extract_table_of_contents(toc)

else:
    print("No Table of Contents found")

In [ ]:
docs = []


for doc_num, doc in enumerate(toc_data):
    doc_link = doc["link"]
    cache_path = f"data/toronto/input/zoning_bylaw/{doc_link}"
    print(cache_path)

    if os.path.exists(cache_path):
        print("file exists")
        with open(cache_path, encoding="utf-8") as file:
            html = file.read()
    else:
        print("fetching file")
        response = requests.get(base_url + doc_link)
        html = response.text
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        with open(cache_path, "w", encoding="utf-8") as file:
            file.write(html)

    soup = BeautifulSoup(html, "html.parser")

    content_header = soup.find("h1")
    if content_header is not None and content_header.parent is not None:
        content_section = content_header.parent
        page_content = extract_content_rows(content_section)
        for item in page_content:
            item.update(
                {
                    "doc_id": doc_num,
                    "link": doc_link,
                    "toc_section": doc["section"],
                    "toc_title": doc["title"],
                }
            )
        docs.extend(page_content)

In [ ]:
pd.DataFrame(docs).to_csv("../data/toronto/output/bylaw_processed.csv")

## Plotting geo data


In [ ]:
import geopandas as gpd

In [ ]:
add_pts = gpd.read_file("../data/toronto/input/mapping/address_points_gpkg.gpkg")

## Data cleaning

In [ ]:
add_pts["geometry"] = add_pts["geometry"].buffer(0)  # common trick to fix minor self-intersections

## EDA

In [ ]:
add_pts.columns

In [ ]:
add_pts["WARD_NAME"].value_counts().head(20)
add_pts.drop(columns="geometry").describe(include="all")

In [ ]:
add_pts.plot(column="WARD_NAME", legend=True, figsize=(10, 10), cmap="tab20")

In [ ]:
add_pts